# Calibration multi-start — Optax from many starts

This page takes the best points of an LHS design and runs multi-start
optimisation: every start is one lane of a single compiled program, chunked
with convergence freezing and an automatic learning-rate probe.

The claim to check:

1. Every **converged** start finishes within 0.02 of the true infection rate
   that generated the synthetic observations.

Loss traces per start show the descent; the dashed line is the truth.


In [ ]:
from typing import NamedTuple

import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
    derived_refs,
)
from summer4.epi.calibration import (
    BayesianModel,
    NormalLikelihood,
    Uniform,
    workflow as wf,
)


## Synthetic SIR and a scored design

Build the same SIR fixture as the design notebook, score an LHS, and keep the
best eight starts (plus the rest as a restart reserve).


In [ ]:
class Rates(NamedTuple):
    infection: float
    recovery: float


TRUE_INFECTION = 0.35
TRUE_RECOVERY = 0.1
times = np.array([0.0, 20.0, 40.0, 60.0])

state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
refs = derived_refs(Rates)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], refs.infection))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], refs.recovery))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))
qty = Compartments(where=state["I"])
truth = cm.run(
    {"infection": TRUE_INFECTION, "recovery": TRUE_RECOVERY},
    y0,
    t0=0.0,
    t1=80.0,
    dt=1.0,
    save=SavePlan(requests={"I": SaveRequest(qty, ts=times)}),
    solver="euler",
)
raw = truth["I"].at_times(times).values
obs = np.asarray(raw.data if hasattr(raw, "data") else raw).reshape(-1)
targets = TargetSet(
    targets=(
        Target(
            key="I",
            times=times,
            values=obs,
            quantity=qty,
            likelihood=NormalLikelihood(sd=5.0),
        ),
    )
)
bm = BayesianModel(
    cm,
    {"recovery": TRUE_RECOVERY},
    priors=(Uniform("infection", 0.05, 1.0),),
    targets=targets,
    y0=y0,
    run_kwargs={"t0": 0.0, "t1": 80.0, "dt": 1.0, "solver": "euler"},
)

design = wf.evaluate(bm, wf.lhs(bm, 128, seed=0), batch_size=64)
starts = design.best(8)


## Multi-start Optax with AutoTune

`wf.optimize` probes a learning-rate grid, then runs Adam (with plateau) in
chunks of 25 steps up to 200, freezing converged starts and restarting failures
from the design reserve.


In [ ]:
result = wf.optimize(
    bm,
    starts,
    method=wf.Optax(learning_rate=0.05),
    tuning=wf.AutoTune(
        lr_grid=(1e-3, 1e-2, 1e-1),
        probe_steps=20,
        probe_starts=4,
        patience=2,
        rtol=1e-5,
    ),
    reserve=design,
    chunk_steps=25,
    max_steps=200,
    seed=0,
)
print(f"learning_rate={result.learning_rate}, chunks={result.loss_trace.shape[0]}")
print("converged:", result.converged)
print("fitted infection:", np.asarray(result.candidates.params["infection"]))


## Loss traces per start

Each curve is one start's best loss after every chunk. Converged starts should
flatten; fitted infection rates should sit near the truth line in the assert
below.


In [ ]:
trace = np.asarray(result.loss_trace)
rows = []
for chunk_i in range(trace.shape[0]):
    for start_i in range(trace.shape[1]):
        rows.append(
            {
                "chunk": chunk_i,
                "start": str(start_i),
                "loss": float(trace[chunk_i, start_i]),
            }
        )
trace_df = pd.DataFrame(rows)
fig = trace_df.plot(
    x="chunk",
    y="loss",
    color="start",
    title="Multi-start Optax: loss per chunk",
)
fig.update_layout(xaxis_title="chunk", yaxis_title="best loss")
fig.show()

fitted = np.asarray(result.candidates.params["infection"])
converged = np.asarray(result.converged, dtype=bool)
assert np.any(converged), "expected at least one converged start"
errs = np.abs(fitted[converged] - TRUE_INFECTION)
assert np.all(errs <= 0.02), (
    f"converged starts {fitted[converged]} not within 0.02 of truth {TRUE_INFECTION}"
)
print(f"converged infection errors max={errs.max():.4f} (tol 0.02)")
